# Building Neural Networks with PyTorch

This notebook introduces the end-to-end workflow for training a multi-layer perceptron (MLP) in PyTorch. We will use the handwritten digits dataset from scikit-learn so the notebook runs quickly in a classroom setting.

## Learning goals

By the end of this notebook, you should be able to:
- prepare data for PyTorch with `TensorDataset` and `DataLoader`,
- define an MLP with `nn.Module`,
- train with Adam and cross-entropy loss, and
- visualize learning curves and model performance.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Load a dataset

The digits dataset contains 8x8 grayscale images of handwritten digits. Each example will be flattened into a vector of 64 features before entering the MLP.

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('Class labels:', np.unique(y))

## Visualize a few examples

Even though an MLP does not explicitly use spatial structure, it is still helpful to inspect the original images.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), digits.images[:10], y[:10]):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.tight_layout()

## Split and standardize the data

Neural networks often train more smoothly when input features are standardized. We will create training, validation, and test splits.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

## Build DataLoaders

`DataLoader` handles batching, shuffling, and iteration. This makes the training loop cleaner and more scalable.

In [ ]:
batch_size = 64

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size)

xb, yb = next(iter(train_loader))
print('Batch feature shape:', xb.shape)
print('Batch label shape:', yb.shape)

## Define the MLP

We will use two hidden layers, ReLU activations, and dropout for regularization.

In [ ]:
class DigitsMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_1=128, hidden_2=64, num_classes=10, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = DigitsMLP().to(device)
model

## Loss function and optimizer

`CrossEntropyLoss` is the standard loss for multi-class classification. Adam is a strong default optimizer for small and medium-sized experiments.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

## Helper functions

To keep the training loop organized, we will separate one-epoch training from evaluation.

In [ ]:
def accuracy_from_logits(logits, y_true):
    preds = logits.argmax(dim=1)
    return (preds == y_true).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_examples += X_batch.size(0)

    return total_loss / total_examples, total_correct / total_examples

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item() * X_batch.size(0)
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_examples += X_batch.size(0)

    return total_loss / total_examples, total_correct / total_examples

## Train the network

The core pattern is always the same: forward pass, compute loss, backpropagate, and step the optimizer.

In [ ]:
num_epochs = 30
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.3f} | val_acc={val_acc:.3f}"
        )

## Plot the learning curves

Loss curves help diagnose optimization behavior, while accuracy curves show how predictive performance changes over time.

In [ ]:
epochs = np.arange(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_loss'], label='Train loss')
axes[0].plot(epochs, history['val_loss'], label='Validation loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train accuracy')
axes[1].plot(epochs, history['val_acc'], label='Validation accuracy')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()

## Evaluate on the held-out test set

Once the model looks reasonable on the validation set, we estimate final performance on the test split.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.3f}')

## Confusion matrix and classification report

These summaries show which digits the model confuses most often.

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device))
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

cm = confusion_matrix(all_targets, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')
plt.title('Digits MLP confusion matrix')
plt.show()

print(classification_report(all_targets, all_preds))

## Wrap-up

This notebook demonstrated a complete PyTorch pipeline for an MLP: prepare tensors, build the model, train with mini-batches, and evaluate using both curves and confusion matrices. The same skeleton will be reused throughout this module.